In [150]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
from datetime import date

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [151]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [152]:
d = date.today().strftime("%Y%m%d")

In [153]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

# Create Feature Class for old pipeline data

In [154]:
# old_parcels = pd.DataFrame.spatial.from_featureclass(r'E:\Projects\REMM-Manage-Base-Year-Data\Current_Inputs\remm_base_year.gdb\parcels')[['parcel_id', 'SHAPE']]
# pipeline = pd.read_csv(r"E:\Projects\REMM-Manage-Base-Year-Data\Current_Inputs\pipeline_buildings_20230405.csv")

In [155]:
# pipeline.columns

In [156]:
# pipeline_polygons = old_parcels.merge(pipeline, left_on='parcel_id', right_on= 'parcel_id', how='right')
# pipeline_polygons.spatial.to_featureclass(location=os.path.join(gdb,"pipeline_polygons_OLD"),sanitize_columns=False) 
# wf_pts_export = arcpy.management.FeatureToPoint(os.path.join(gdb,"pipeline_polygons_OLD"), os.path.join(gdb,"pipeline_pts_OLD"), "INSIDE")

# Create Pipeline Data from new parcels

In [157]:
pcls = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels')

In [158]:
####################################################
# Fill missing  sqft values (this needs to account for building type specifcally sf vs rest)
####################################################

buildings = pcls[(pcls['building_type_id'].isin([1,2,3,4,5,6,7,8]))].copy()

buildings_for_assumptions = buildings[(buildings['building_sqft'] > 0) & (buildings['parcel_acres'] > 0)].copy()
buildings_for_assumptions['building_sqft_per_acre'] = buildings_for_assumptions['building_sqft'] / buildings_for_assumptions['parcel_acres']

building_median_sqft_acre_medium_district = buildings_for_assumptions.groupby(['distmed_id', 'building_type_id'], as_index=False)[['building_sqft_per_acre']].median()
building_median_sqft_acre_medium_district.columns = ['distmed_id', 'building_type_id', 'median_building_sqft_per_acre_MD']
buildings = buildings.merge(building_median_sqft_acre_medium_district, on=['distmed_id', 'building_type_id'], how='left')

# Mask for invalid building_sqft 
mask = ((buildings['building_sqft'].isna()) |  (buildings['building_sqft'].isin([0, -9999]))) 

# Fill with  values for medium district
buildings.loc[mask, 'building_sqft'] = round(buildings['parcel_acres'] * buildings['median_building_sqft_per_acre_MD'])
del buildings['median_building_sqft_per_acre_MD']


# second pass with large district values
building_median_sqft_acre_large_district = buildings_for_assumptions.groupby(['distlrg_id', 'building_type_id'], as_index=False)[['building_sqft_per_acre']].median()
building_median_sqft_acre_large_district.columns = ['distlrg_id', 'building_type_id', 'median_building_sqft_per_acre_LD']
buildings = buildings.merge(building_median_sqft_acre_large_district, on=['distlrg_id', 'building_type_id'], how='left')

# Mask for invalid building_sqft 
mask = ((buildings['building_sqft'].isna()) |  (buildings['building_sqft'].isin([0, -9999]))) 

# Fill with  values for medium district
buildings.loc[mask, 'building_sqft'] = round(buildings['parcel_acres'] * buildings['median_building_sqft_per_acre_LD'])
del buildings['median_building_sqft_per_acre_LD']

# third pass with county values
building_median_sqft_acre_county = buildings_for_assumptions.groupby(['county_id', 'building_type_id'], as_index=False)[['building_sqft_per_acre']].median()
building_median_sqft_acre_county.columns = ['county_id', 'building_type_id', 'median_building_sqft_per_acre_CNTY']
buildings = buildings.merge(building_median_sqft_acre_county, on=['county_id', 'building_type_id'], how='left')

# Mask for invalid building_sqft 
mask = ((buildings['building_sqft'].isna()) |  (buildings['building_sqft'].isin([0, -9999]))) 

# Fill with  values for medium district
buildings.loc[mask, 'building_sqft'] = round(buildings['parcel_acres'] * buildings['median_building_sqft_per_acre_CNTY'])
del buildings['median_building_sqft_per_acre_CNTY']

# convert to int
buildings['building_sqft'] = buildings['building_sqft'].astype(int)

In [159]:
pcls_pipeline = buildings[buildings['year_built'] > 2023].copy()
pcls_pipeline.head()

,OBJECTID,parcel_id,WFRC_parcel_id,county_id,CO_NAME,year_built,total_market_value,land_value,building_id,building_type_id,building_type,building_sqft,non_residential_sqft,residential_units,job_spaces,stories,unit_price_non_residential,res_price_per_sqft,basebldg,redev_friction,NoBuild,IS_OUG,parcel_acres,Tax_Exempt,parent_parcel,volume_one_way,volume_two_way,volume_two_way_nofwy,zonal_ppa,x,y,note,parcel_sqft,Split,Split_Factor,MAG_parcel_id,max_far,max_dua,type1,type2,type3,type4,type5,type6,type7,type8,TAZID_900,distsml_id,distmed_id,distlrg_id,CITY_NAME,stream_dist,streams,trail_dist,trail,airport_distance,airport,fwy_exit_dist,fwy_exit_new,bus_stop_dist_new,bus_stop_new,bus_rte_dist,rail_stn_dist,rail_stn_new,raildepot_dist,rail_depot,university_dist,university,elevation,agriculture,TAZID_910,grid_id,SHAPE
13,2238,2237,2237,57,WEBER,2024,387844,387844,2237,6,Agriculture,15000,238,0,20,0,-9999,-9999.0,1,-9999,0,0,5.216221,1,550348,105,207,207,183882.6035,412512.935379,4564576.688485,base,1136784.215966,1,,,0.5,<NA>,0,0,0,0,0,1,0,0,450,13,11,4,West Haven,97.988445,0,84.187088,0,4414.730692,0,2633.721448,0,1521.636462,0,1531.109849,4358.112425,0,52802.90587,0,5057.492982,0,1298,0,450,102105,"{""rings"": [[[412445.6979, 4564484.157299999], ..."
14,2239,2238,2238,57,WEBER,2024,387844,387844,2238,6,Agriculture,15000,238,0,20,0,-9999,-9999.0,1,-9999,0,0,5.216222,1,550348,105,207,207,183882.6035,412663.715525,4564594.401822,base,1136784.215966,1,,,0.5,<NA>,0,0,0,0,0,1,0,0,450,13,11,4,West Haven,77.405996,0,66.601246,0,4344.154612,0,2482.441848,0,1373.699575,0,1380.692657,4367.237866,0,52789.623225,0,4935.363125,0,1298,0,450,102106,"{""rings"": [[[412582.22470000014, 4564655.50699..."
15,2240,2239,2239,57,WEBER,2024,387844,387844,2239,6,Agriculture,15000,238,0,20,0,-9999,-9999.0,1,-9999,0,0,5.216223,1,550348,105,207,207,183882.6035,412501.797212,4564758.532483,base,1136784.215966,1,,,0.5,<NA>,0,0,0,0,0,1,0,0,450,13,11,4,West Haven,278.537859,0,265.32656,0,4570.682301,0,2645.355536,0,1561.172061,0,1547.768025,4540.338113,0,52983.246376,0,4944.462869,0,1299,0,450,102302,"{""rings"": [[[412449.01049999986, 4564655.50699..."
16,2241,2240,2240,57,WEBER,2024,387844,387844,2240,6,Agriculture,15000,238,0,20,0,-9999,-9999.0,1,-9999,0,0,5.216223,1,550348,105,207,207,183882.6035,412604.207653,4564757.928538,base,1136784.215966,1,,,0.5,<NA>,0,0,0,0,0,1,0,0,450,13,11,4,West Haven,251.428545,0,240.600285,0,4513.589736,0,2542.963883,0,1461.111927,0,1445.539968,4533.345225,0,52961.850291,0,4868.032492,0,1299,0,450,102302,"{""rings"": [[[412552.7909000004, 4564861.653200..."
17,2242,2241,2241,57,WEBER,2024,387844,387844,2241,6,Agriculture,15000,238,0,20,0,-9999,-9999.0,1,-9999,0,0,5.216223,1,550348,105,207,207,183882.6035,412707.932328,4564757.266544,base,1136784.215966,1,,,0.5,<NA>,0,0,0,0,0,1,0,0,450,13,11,4,West Haven,213.774929,0,201.715684,0,4457.376731,0,2439.260799,0,1360.117348,0,1342.029714,4528.564178,0,52940.323655,0,4791.645382,0,1298,0,450,102303,"{""rings"": [[[412655.84339999966, 4564859.0414]..."


In [160]:
pcls_pipeline = pcls_pipeline[['parcel_id', 'building_type_id', 'year_built','building_sqft', 'residential_units', 'job_spaces', 'county_id']]
pcls_pipeline['DEVTYPE'] = 'develop'
pcls_pipeline['note'] = ''
pcls_pipeline['source'] = 'assessor or aerials'
pcls_pipeline['TAZID_900'] = np.nan
pcls_pipeline['unit_price_non_residential'] = 0
pcls_pipeline['res_price_per_sqft'] = 0
pcls_pipeline['residential_price'] = 0
pcls_pipeline['non_residential_price'] = 0
pcls_pipeline['stories'] = 1
pcls_pipeline['non_residential_sqft'] = 0
pcls_pipeline.loc[pcls_pipeline['building_type_id'].isin([3,4,5,6,7,8]), 'non_residential_sqft'] = pcls_pipeline['building_sqft']
pcls_pipeline.loc[pcls_pipeline['county_id'].isin([3, 57, 11, 35]), 'agency'] = 'WFRC'
pcls_pipeline.loc[pcls_pipeline['county_id'].isin([49]), 'agency'] = 'MAG'

In [161]:
old_pipeline = pd.DataFrame.spatial.from_featureclass(os.path.join(gdb,"pipeline_pts_OLD"))
old_pipeline.head(2)

,OBJECTID,parcel_id,building_type_id,non_residential_sqft,note,pipeline_id,residential_units,stories,unit_price_non_residential,year_built,res_price_per_sqft,residential_price,non_residential_price,job_spaces,building_sqft,DEVTYPE,source,TAZID_900,agency,ORIG_FID,SHAPE
0,1,57047,2,0,pipeline,0,301,2,0,2024,0,0,0,0,481600,develop,WVC,1437,WFRC,1,"{""x"": 412993.85809999984, ""y"": 4498937.0239, ""..."
1,2,57339,2,0,pipeline,1,89,2,0,2024,0,0,0,0,106800,develop,WVC,1437,WFRC,2,"{""x"": 412970.6878000004, ""y"": 4498610.66169999..."


In [162]:
olivia_pipeline = pd.DataFrame.spatial.from_featureclass(r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Pipeline_Projects.gdb\pipeline_buildings_pts_RTP2027')
old_pipeline.head(2)

,OBJECTID,parcel_id,building_type_id,non_residential_sqft,note,pipeline_id,residential_units,stories,unit_price_non_residential,year_built,res_price_per_sqft,residential_price,non_residential_price,job_spaces,building_sqft,DEVTYPE,source,TAZID_900,agency,ORIG_FID,SHAPE
0,1,57047,2,0,pipeline,0,301,2,0,2024,0,0,0,0,481600,develop,WVC,1437,WFRC,1,"{""x"": 412993.85809999984, ""y"": 4498937.0239, ""..."
1,2,57339,2,0,pipeline,1,89,2,0,2024,0,0,0,0,106800,develop,WVC,1437,WFRC,2,"{""x"": 412970.6878000004, ""y"": 4498610.66169999..."


In [163]:
new_pipeline = pd.concat([old_pipeline, pcls_pipeline, olivia_pipeline])
new_pipeline

,OBJECTID,parcel_id,building_type_id,non_residential_sqft,note,pipeline_id,residential_units,stories,unit_price_non_residential,year_built,res_price_per_sqft,residential_price,non_residential_price,job_spaces,building_sqft,DEVTYPE,source,TAZID_900,agency,ORIG_FID,SHAPE,county_id
0,1,57047,2,0,pipeline,0,301,2,0,2024,0,0,0,0,481600,develop,WVC,1437,WFRC,1,"{""x"": 412993.85809999984, ""y"": 4498937.0239, ""...",<NA>
1,2,57339,2,0,pipeline,1,89,2,0,2024,0,0,0,0,106800,develop,WVC,1437,WFRC,2,"{""x"": 412970.6878000004, ""y"": 4498610.66169999...",<NA>
2,3,34472,2,0,pipeline,2,100,3,0,2025,0,0,0,0,160000,develop,WVC,1423,WFRC,3,"{""x"": 418921.52610000037, ""y"": 4502556.3008999...",<NA>
3,7,64911,2,0,pipeline,6,329,6,0,2025,0,0,0,0,400000,develop,WVC,1383,WFRC,7,"{""x"": 418663.29389999993, ""y"": 4505175.5092999...",<NA>
4,9,28716,5,587000,pipeline,8,0,1,0,2027,0,0,0,2250,587000,develop,WVC: UofU Hospital,1374,WFRC,9,"{""x"": 412730.2965000002, ""y"": 4504232.05320000...",<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,16671,<NA>,2,<NA>,pipeline,<NA>,147,<NA>,<NA>,2026,<NA>,<NA>,<NA>,<NA>,<NA>,redevelop,BSL,<NA>,WFRC,<NA>,"{""x"": 422427.5179000003, ""y"": 4513915.0099, ""s...",35
81,16672,<NA>,2,<NA>,pipeline,<NA>,218,<NA>,<NA>,2027,<NA>,<NA>,<NA>,<NA>,<NA>,redevelop,BSL,<NA>,WFRC,<NA>,"{""x"": 423503.6010999996, ""y"": 4512164.6589, ""s...",35
82,16673,<NA>,2,<NA>,pipeline,<NA>,283,<NA>,<NA>,2028,<NA>,<NA>,<NA>,<NA>,<NA>,redevelop,BSL,<NA>,WFRC,<NA>,"{""x"": 420305.76580000017, ""y"": 4513758.4223, ""...",35
83,16674,<NA>,7,9915,pipeline,<NA>,8,3,<NA>,2027,<NA>,<NA>,<NA>,20,<NA>,redevelop,BSL,<NA>,WFRC,<NA>,"{""x"": 429473.6786000002, ""y"": 4501357.56519999...",35


In [164]:
new_pipeline['DEVTYPE'].value_counts()

DEVTYPE
develop       11709
redevelop        92
demolition        4
Name: count, dtype: int64

In [165]:
new_pipeline['non_residential_sqft'] = new_pipeline['non_residential_sqft'].fillna(0)
new_pipeline['res_price_per_sqft'] = new_pipeline['res_price_per_sqft'].fillna(0)
new_pipeline['unit_price_non_residential'] = new_pipeline['unit_price_non_residential'].fillna(0)
new_pipeline['residential_price'] = new_pipeline['residential_price'].fillna(0)
new_pipeline['non_residential_price'] = new_pipeline['non_residential_price'].fillna(0)
new_pipeline['job_spaces'] = new_pipeline['job_spaces'].fillna(0)
new_pipeline['building_sqft'] = new_pipeline['building_sqft'].fillna(0)
new_pipeline['residential_units'] = new_pipeline['residential_units'].fillna(0)
new_pipeline['stories'] = new_pipeline['stories'].fillna(0)

In [166]:
# cast columns
new_pipeline['non_residential_sqft'] = new_pipeline['non_residential_sqft'].astype(int)
new_pipeline['building_type_id'] = new_pipeline['building_type_id'].astype(int)
new_pipeline['year_built'] = new_pipeline['year_built'].astype(int)
new_pipeline['residential_units'] = new_pipeline['residential_units'].astype(int)
new_pipeline['job_spaces'] = new_pipeline['job_spaces'].astype(int)
new_pipeline['non_residential_price'] = new_pipeline['non_residential_price'].astype(int)
new_pipeline['unit_price_non_residential'] = new_pipeline['unit_price_non_residential'].astype(int)

In [167]:
# subset columns
new_pipeline = new_pipeline[['building_type_id','non_residential_sqft','note',
                                         'residential_units','stories','unit_price_non_residential','year_built',
                                         'res_price_per_sqft','residential_price','non_residential_price','job_spaces',
                                         'building_sqft','DEVTYPE', 'SHAPE']].copy()


# export

In [168]:
new_pipeline.spatial.to_featureclass(location=os.path.join(gdb,"pipeline_polygons_NEW"),sanitize_columns=False) 
new_pipeline_pts = arcpy.management.FeatureToPoint(os.path.join(gdb,"pipeline_polygons_NEW"), os.path.join(gdb, 'pipeline_pts_no_parcel_id'), "INSIDE")

In [169]:
# spatial join
target_features = new_pipeline_pts
join_features = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_ids_only'
output_features =  os.path.join(gdb, 'pipeline_pts_sj')

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

del sj_df['ORIG_FID']
del sj_df['TARGET_FID']
del sj_df['Join_Count']
del sj_df['OBJECTID']

In [170]:
sj_df.spatial.to_featureclass(location=r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Pipeline_Projects.gdb\pipeline_buildings_pts_RTP2027_20260125',sanitize_columns=False) 
sj_df.drop('SHAPE', axis=1).to_csv(os.path.join(outputs[0], 'pipeline_buildings_20260125.csv'), index=False)